# Gold | Construção das dimensões (em `gold_staging`)

As dimensões são construídas em **`workspace.gold_staging`**, validadas pelo DQ Gold (Job 07) e só então promovidas
para `workspace.gold` (`Gold/Qualidade_dados/03_promover_gold`). Assim uma falha bloqueante não deixa dado ruim publicado.

| Dimensão | Origem |
|---|---|
| `dim_tempo` | `silver.municipio` |
| `dim_rede`, `dim_serie` | `silver.dicionario` (tradução oficial; sem códigos inventados) |
| `dim_uf` | `silver.diretorio_uf` (Base dos Dados) — não depende mais de CSV em Volume |
| `dim_municipio` | `silver.municipio` + `dim_uf` (UF pelos 2 primeiros dígitos do código IBGE) + `silver.diretorio_municipio` |
| `dim_situacao_meta` | estados usados nas fatos |

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
_m = iniciar_etapa()
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold_staging")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_staging.dim_tempo USING DELTA AS
SELECT DISTINCT ano, make_date(ano, 1, 1) AS data_referencia
FROM workspace.silver.municipio
WHERE ano IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_staging.dim_rede USING DELTA AS
SELECT DISTINCT CAST(chave AS STRING) AS rede_codigo, TRIM(valor) AS descricao_rede
FROM workspace.silver.dicionario
WHERE id_tabela = 'municipio' AND nome_coluna = 'rede' AND chave IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_staging.dim_serie USING DELTA AS
SELECT DISTINCT CAST(chave AS INT) AS serie_codigo, TRIM(valor) AS serie_descricao
FROM workspace.silver.dicionario
WHERE id_tabela = 'municipio' AND nome_coluna = 'serie' AND chave IS NOT NULL;

In [0]:
%sql
-- dim_uf ANTES de dim_municipio (dim_municipio depende dela)
CREATE OR REPLACE TABLE workspace.gold_staging.dim_uf USING DELTA AS
SELECT id_uf, sigla_uf, nome_uf, regiao
FROM workspace.silver.diretorio_uf;

In [0]:
%sql
-- UF = 2 primeiros dígitos do código IBGE do município; nome vem do diretório oficial.
-- Município sem correspondência fica com nome/UF NULOS (não é preenchido) e é medido pelo DQ Gold.
CREATE OR REPLACE TABLE workspace.gold_staging.dim_municipio USING DELTA AS
SELECT
    m.id_municipio,
    u.id_uf,
    u.sigla_uf,
    u.nome_uf,
    u.regiao,
    d.nome_municipio
FROM (SELECT DISTINCT id_municipio FROM workspace.silver.municipio WHERE id_municipio IS NOT NULL) m
LEFT JOIN workspace.gold_staging.dim_uf u
    ON CAST(SUBSTRING(m.id_municipio, 1, 2) AS INT) = u.id_uf
LEFT JOIN workspace.silver.diretorio_municipio d
    ON m.id_municipio = d.id_municipio;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_staging.dim_situacao_meta USING DELTA AS
SELECT * FROM VALUES
    ('ATINGIU_OU_SUPEROU', 'Resultado >= meta'),
    ('ABAIXO_DA_META', 'Resultado < meta'),
    ('SEM_META', 'Existe resultado, mas não há meta cadastrada para o recorte'),
    ('SEM_RESULTADO', 'Existe meta, mas o resultado é nulo'),
    ('SEM_META_DISPONIVEL_PARA_COMPARACAO', 'Ano do resultado fora da trajetória de metas')
AS t(situacao_meta, descricao);

In [0]:
for t in ["dim_tempo", "dim_rede", "dim_serie", "dim_uf", "dim_municipio", "dim_situacao_meta"]:
    n = contar_tabela(f"workspace.gold_staging.{t}")
    registrar_execucao("GOLD", t, _m, row_count=n)
    print(t, n)